# MatrixDB on Google Colab

GPU-accelerated database engine: page-ownership point ops + a resident-column `u32x4` scan kernel, behind a lock-free SPSC ring + dual-trigger batcher.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

This notebook writes its own source files (cells below), runs the CPU coverage test, builds with `nvcc`, and runs. No uploads needed. Success ends with `Scan result sum: 83886070 (oracle 83886070)` — asserts fire on any mismatch.

## 1. Confirm a GPU is attached

In [ ]:
!nvcc --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Write the source files

In [ ]:
%%writefile types.hpp
#pragma once

#include <cstdint>
#include <cstddef> // ponytail: spec wrote <csize_t> (does not exist); size_t lives in <cstddef>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Apple Silicon L1 cache lines are typically 128 bytes
    constexpr size_t MATRIX_CACHE_LINE = 128;
#else
    // Standard Intel/AMD x86_64 cache lines are 64 bytes
    constexpr size_t MATRIX_CACHE_LINE = 64;
#endif

/**
 * @brief Represents a single transaction query record.
 * Designed with descending structural member alignment to prevent internal compiler padding.
 * Aligned to target CPU cache line boundaries to eliminate false sharing.
 */
struct alignas(MATRIX_CACHE_LINE) DatabaseQuery {
    uint64_t query_id;
    uint64_t timestamp_us;
    uint64_t transaction_id;
    uint32_t opcode;
    uint32_t payload_size;
    uint8_t payload[32];

    static constexpr size_t data_footprint =
        sizeof(uint64_t) * 3 +
        sizeof(uint32_t) * 2 +
        sizeof(uint8_t) * 32;

    static constexpr size_t padding_needed =
        (MATRIX_CACHE_LINE > data_footprint) ? (MATRIX_CACHE_LINE - data_footprint) : 0;

    // Explicit structural padding to align with target L1 cache line size
    uint8_t padding[padding_needed > 0 ? padding_needed : 1];
};

static_assert((sizeof(DatabaseQuery) % MATRIX_CACHE_LINE) == 0,
    "DatabaseQuery structure violates cache-line alignment constraints.");

// Operational instruction carried in DatabaseQuery::opcode (spec: Read, Write, Scan).
enum MatrixOpcode : uint32_t {
    OP_READ  = 1,
    OP_WRITE = 2,
    OP_SCAN  = 3,
};

// Columnar store + page-ownership layout (Dragonfly-style shard-per-thread).
// The keyspace (STORE_SLOTS) is split into PAGE_COUNT contiguous pages. Exactly one
// owner (one GPU thread) reads/writes a page's slots, so the same key always routes
// to the same owner: writes to a key serialize by ownership. No cross-thread conflict
// => no store atomics, no OCC, no delta log. Per page it is single-owner (Redis);
// across pages it is shared-nothing (Dragonfly).
constexpr size_t MATRIX_STORE_SLOTS = 4096;                       // power of two
constexpr size_t MATRIX_STORE_MASK  = MATRIX_STORE_SLOTS - 1;
constexpr size_t MATRIX_PAGE_COUNT  = 1024;                       // owner threads; the tuning knob
constexpr size_t MATRIX_PAGE_SIZE   = MATRIX_STORE_SLOTS / MATRIX_PAGE_COUNT; // slots per page
constexpr size_t MATRIX_BATCH_MAX   = 65536;                      // sweep ceiling / scratch buffer size

// Resident analytical column (the GPU-DB's actual data): a uint32 column that lives
// in VRAM/RAM and is scanned in place by OP_SCAN — never shipped per query. 16M values
// = 64MB, well past CPU cache so the GPU bandwidth advantage is real. Filled value[i]=i
// so a threshold T yields a known count (SCAN_SIZE-1-T) for oracle verification.
constexpr size_t MATRIX_SCAN_COLUMN_SIZE = 1u << 24;             // 16,777,216 values (divisible by 4 for uint4)

static_assert(MATRIX_STORE_SLOTS % MATRIX_PAGE_COUNT == 0,
    "store slots must divide evenly into pages so each page owns a contiguous slice");


In [ ]:
%%writefile ring_buffer.hpp
#pragma once

#include "types.hpp"
#include <atomic>
#include <array>
#include <utility>

/**
 * @brief Ultra-low latency, lock-free, zero-allocation SPSC Ring Buffer.
 * Enforces power-of-two capacities to replace expensive modulo operations with bitwise AND masking.
 */
template <typename T, size_t Capacity>
class SPSCRingBuffer {
    static_assert((Capacity & (Capacity - 1)) == 0, "SPSCRingBuffer Capacity must be a power of two.");
    static_assert(Capacity >= 2, "SPSCRingBuffer Capacity must be at least two elements.");

public:
    SPSCRingBuffer()
        : head_(0)
        , tail_(0)
        , cached_head_(0)
        , cached_tail_(0) {}

    ~SPSCRingBuffer() = default;

    SPSCRingBuffer(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer& operator=(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer(SPSCRingBuffer&&) = delete;
    SPSCRingBuffer& operator=(SPSCRingBuffer&&) = delete;

    /**
     * @brief Emplaces a new item at the tail of the buffer. Only the Producer may call this method.
     */
    template <typename... Args>
    bool try_emplace(Args&&... args) {
        const size_t current_tail = tail_.load(std::memory_order_relaxed);

        // Check local cached head before loading the atomic head_ cursor from shared memory
        if (current_tail - cached_head_ >= Capacity) {
            cached_head_ = head_.load(std::memory_order_acquire);
            if (current_tail - cached_head_ >= Capacity) {
                return false; // Queue is genuinely full
            }
        }

        const size_t index = current_tail & MASK;
        buffer_[index] = T{std::forward<Args>(args)...};

        tail_.store(current_tail + 1, std::memory_order_release);
        return true;
    }

    /**
     * @brief Pops an item from the head of the buffer. Only the Consumer may call this method.
     */
    bool try_pop(T& value) {
        const size_t current_head = head_.load(std::memory_order_relaxed);

        // Check local cached tail before loading the atomic tail_ cursor from shared memory
        if (current_head >= cached_tail_) {
            cached_tail_ = tail_.load(std::memory_order_acquire);
            if (current_head >= cached_tail_) {
                return false; // Queue is genuinely empty
            }
        }

        const size_t index = current_head & MASK;
        value = std::move(buffer_[index]);

        head_.store(current_head + 1, std::memory_order_release);
        return true;
    }

    [[nodiscard]] bool empty() const noexcept {
        return head_.load(std::memory_order_relaxed) == tail_.load(std::memory_order_relaxed);
    }

    [[nodiscard]] size_t size() const noexcept {
        const size_t t = tail_.load(std::memory_order_relaxed);
        const size_t h = head_.load(std::memory_order_relaxed);
        return (t >= h) ? (t - h) : 0;
    }

private:
    static constexpr size_t MASK = Capacity - 1;

    // Isolate variables on separate cache lines to eliminate false sharing
    alignas(MATRIX_CACHE_LINE) std::array<T, Capacity> buffer_{};
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> head_;
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> tail_;

    // Thread-local shadow indices
    alignas(MATRIX_CACHE_LINE) size_t cached_head_;
    alignas(MATRIX_CACHE_LINE) size_t cached_tail_;
};


In [ ]:
%%writefile compute.hpp
#pragma once

#include "types.hpp"
#include <cstddef>
#include <cstdint>

/**
 * @brief Pure virtual interface defining the compute engine's entry point.
 * Enables zero-copy, raw pointer processing over batch slices.
 *
 * Two implementations satisfy this contract and are selected at compile time:
 *   - CPUMockEngine (compute_mock.cpp)  — default, runs anywhere, no GPU needed
 *   - CUDAGPUEngine (compute_cuda.cuh)  — built with nvcc when MATRIX_USE_CUDA is set
 *
 * Both use the page-ownership model: a key always routes to one owning page, so the
 * two backends produce byte-identical store contents (verifiable in main.cpp).
 */
class ComputeInterface {
public:
    virtual ~ComputeInterface() = default;
    virtual void execute_batch(DatabaseQuery* batch_array, size_t count) = 0;

    virtual uint64_t reads() const = 0;
    virtual uint64_t writes() const = 0;
    virtual uint64_t commits() const = 0; // writes actually applied to the store

    // OP_SCAN result accounting: how many scan queries ran, and the running sum of their
    // filter-count results. main asserts the sum against a known oracle to prove scans
    // flowed ring -> batcher -> execute_batch -> (GPU) scan kernel correctly.
    virtual uint64_t scans() const = 0;
    virtual uint64_t scan_result_sum() const = 0;

    // Wall time the engine spent inside OP_SCAN work. Lets the pipeline report point-op
    // and scan throughput separately — a single 64MB scan costs ~1000x a point op, so
    // one combined "ops/sec" number is meaningless once scans are mixed in.
    virtual double scan_time_s() const = 0;

    // Order-independent fingerprint of the whole store (sum of slot values). Lets
    // main.cpp assert CPU and GPU reach byte-identical state — the real test of the
    // ownership model, not just matching counts.
    virtual uint64_t store_checksum() const = 0;

    // Scan benchmark: allocate n resident values (value[i]=i), then time a single
    // filter-count scan (count of value[i] > threshold) over that resident data.
    // alloc+fill are NOT timed — the point is "data lives here, query scans it".
    // This is the GPU's home turf (bandwidth over data too big for CPU cache),
    // the opposite of the point-op path. Returns seconds; sets out_count.
    // out_count is deterministic (value[i]=i) so it must match across CPU and GPU.
    virtual double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) = 0;

    // Same scan over a uint32 column. Scan is bandwidth-bound, so halving bytes/value
    // should ~double values/sec at the same GB/s — the columnar "narrowest type" win.
    // If GB/s holds vs the uint64 scan, we are at the bandwidth wall (vectorized loads
    // won't help); if it drops, narrower loads underfill the bus and vectorizing is next.
    virtual double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) = 0;

    // Vectorized uint32 scan (4 values per load via uint4). Tests the memory-level
    // parallelism theory: if this beats the scalar u32 GB/s, narrow scalar loads were
    // underfilling the bus (MLP-bound) and wider loads are the fix. If it doesn't, the
    // kernel is ALU-bound and we stop optimizing. CPU may treat this same as scalar.
    virtual double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) = 0;

    // Items-per-thread scan (register blocking, ITEMS independent loads/thread before
    // comparing). CUB's BlockReduce lever. Tests whether deeper memory-level parallelism
    // per thread beats both scalar and uint4. CPU delegates (compiler auto-vectorizes).
    virtual double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) = 0;
};

// Which page owns a query's key. Single source of truth for both engines and the
// kernel: slot = key & STORE_MASK, page = slot / PAGE_SIZE.
inline size_t matrix_page_of(uint64_t key) {
    return (key & MATRIX_STORE_MASK) / MATRIX_PAGE_SIZE;
}

// OP_SCAN carries its filter threshold in the query payload (that's what payload is for).
// One codec used by both engines so they decode identically.
inline void matrix_set_scan_threshold(DatabaseQuery& q, uint32_t threshold) {
    q.opcode = OP_SCAN;
    *reinterpret_cast<uint32_t*>(q.payload) = threshold;
}
inline uint32_t matrix_get_scan_threshold(const DatabaseQuery& q) {
    return *reinterpret_cast<const uint32_t*>(q.payload);
}

/**
 * @brief Stable counting-sort of a batch by owning page (the "bin in the batcher" step).
 * Fills `binned` with the queries grouped by page — order within a page preserved, so
 * same-slot writes keep batch order and last-writer-wins is deterministic. `offsets`
 * is a CSR index: page p's queries are binned[offsets[p] .. offsets[p+1]).
 * Caller owns the buffers (binned >= count, offsets >= PAGE_COUNT+1) — zero alloc here.
 */
inline void matrix_bin_by_page(const DatabaseQuery* batch, size_t count,
                               DatabaseQuery* binned, uint32_t* offsets) {
    uint32_t counts[MATRIX_PAGE_COUNT] = {0};
    for (size_t i = 0; i < count; ++i) {
        counts[matrix_page_of(batch[i].query_id)]++;
    }
    offsets[0] = 0;
    for (size_t p = 0; p < MATRIX_PAGE_COUNT; ++p) {
        offsets[p + 1] = offsets[p] + counts[p];
    }
    uint32_t cursor[MATRIX_PAGE_COUNT];
    for (size_t p = 0; p < MATRIX_PAGE_COUNT; ++p) cursor[p] = offsets[p];
    for (size_t i = 0; i < count; ++i) {
        const size_t p = matrix_page_of(batch[i].query_id);
        binned[cursor[p]++] = batch[i];
    }
}


In [ ]:
%%writefile compute_mock.cpp
#include "compute.hpp"
#include <vector>
#include <array>
#include <chrono>
#include <iostream>

/**
 * @brief CPU Mock Engine — the no-GPU fallback (Component 5: Local Sandbox).
 * Page-ownership model: bins the batch by owning page, then processes each page's
 * queries against its own slice of the store. One owner per page ⇒ no atomics, no
 * delta log, deterministic last-writer-wins. Mirrors the CUDA engine's semantics so
 * both produce identical store contents.
 */
class CPUMockEngine : public ComputeInterface {
public:
    explicit CPUMockEngine(size_t /*worker_count*/ = 0)
        : binned_(MATRIX_BATCH_MAX)
        , scan_column_(MATRIX_SCAN_COLUMN_SIZE) {
        for (size_t i = 0; i < MATRIX_SCAN_COLUMN_SIZE; ++i)
            scan_column_[i] = static_cast<uint32_t>(i); // resident analytical column
        std::cout << "CPUMockEngine initialized (page-ownership, "
                  << MATRIX_PAGE_COUNT << " pages, "
                  << MATRIX_SCAN_COLUMN_SIZE << "-value scan column)." << std::endl;
    }

    ~CPUMockEngine() override {
        std::cout << "CPUMockEngine shutdown cleanly." << std::endl;
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX;

        // Bin by page (the step the dual-trigger batcher will eventually fold in).
        matrix_bin_by_page(batch_array, count, binned_.data(), offsets_.data());

        // One logical owner per page: process only this page's queries against its
        // contiguous store slice. Pages are independent — this is the parallel unit.
        for (size_t page = 0; page < MATRIX_PAGE_COUNT; ++page) {
            const uint32_t begin = offsets_[page];
            const uint32_t end = offsets_[page + 1];
            for (uint32_t j = begin; j < end; ++j) {
                DatabaseQuery& q = binned_[j];
                const size_t slot = q.query_id & MATRIX_STORE_MASK;
                if (q.opcode == OP_READ) {
                    q.transaction_id = store_[slot];
                    ++reads_;
                } else if (q.opcode == OP_WRITE) {
                    store_[slot] = q.query_id; // mock projection: value == key
                    ++writes_;
                    ++commits_;
                } else if (q.opcode == OP_SCAN) {
                    // Scan the resident column for values > threshold (filter-count).
                    // ponytail: one full scan per scan query. Batching predicates into a
                    // single pass is the future optimization; correctness first.
                    const uint32_t threshold = matrix_get_scan_threshold(q);
                    const auto st0 = std::chrono::steady_clock::now();
                    uint64_t c = 0;
                    for (size_t s2 = 0; s2 < MATRIX_SCAN_COLUMN_SIZE; ++s2)
                        c += (scan_column_[s2] > threshold);
                    scan_time_s_ += std::chrono::duration<double>(
                        std::chrono::steady_clock::now() - st0).count();
                    q.transaction_id = c;
                    ++scans_;
                    scan_result_sum_ += c;
                }
            }
        }

        // ponytail: read results land in binned_ (reordered), not scattered back to
        // batch_array. Callers here assert on counters + store contents, not on each
        // query's transaction_id, so the scatter-back is YAGNI. Add it if a caller
        // needs per-query read results in original order.
    }

    uint64_t reads() const override { return reads_; }
    uint64_t writes() const override { return writes_; }
    uint64_t commits() const override { return commits_; }
    uint64_t scans() const override { return scans_; }
    uint64_t scan_result_sum() const override { return scan_result_sum_; }
    double scan_time_s() const override { return scan_time_s_; }

    uint64_t store_checksum() const override {
        uint64_t sum = 0;
        for (uint64_t v : store_) sum += v;
        return sum;
    }

    double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) override {
        std::vector<uint64_t> data(n);
        for (size_t i = 0; i < n; ++i) data[i] = i; // resident; fill not timed
        const auto t0 = std::chrono::steady_clock::now();
        uint64_t count = 0;
        for (size_t i = 0; i < n; ++i) count += (data[i] > threshold);
        const auto t1 = std::chrono::steady_clock::now();
        out_count = count;
        return std::chrono::duration<double>(t1 - t0).count();
    }

    double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) override {
        std::vector<uint32_t> data(n);
        for (size_t i = 0; i < n; ++i) data[i] = static_cast<uint32_t>(i);
        const auto t0 = std::chrono::steady_clock::now();
        uint64_t count = 0;
        for (size_t i = 0; i < n; ++i) count += (data[i] > threshold);
        const auto t1 = std::chrono::steady_clock::now();
        out_count = count;
        return std::chrono::duration<double>(t1 - t0).count();
    }

    double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // ponytail: uint4 vectorized loads are a GPU concept; the CPU compiler already
        // auto-vectorizes the scalar loop, so this just delegates. Keeps the interface
        // uniform without faking a "CPU vectorized" path.
        return benchmark_scan_u32(n, threshold, out_count);
    }

    double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // ponytail: register-blocking is a GPU latency-hiding lever; on CPU it's the same
        // auto-vectorized loop. Delegate.
        return benchmark_scan_u32(n, threshold, out_count);
    }

private:
    std::array<uint64_t, MATRIX_STORE_SLOTS> store_{}; // the Value column
    std::vector<DatabaseQuery> binned_;                // scratch: batch sorted by page
    std::array<uint32_t, MATRIX_PAGE_COUNT + 1> offsets_{}; // CSR page offsets
    std::vector<uint32_t> scan_column_;                // resident analytical column
    uint64_t reads_ = 0;
    uint64_t writes_ = 0;
    uint64_t commits_ = 0;
    uint64_t scans_ = 0;
    uint64_t scan_result_sum_ = 0;
    double scan_time_s_ = 0.0;
};


In [ ]:
%%writefile compute_cuda.cuh
#pragma once

// CUDA backend — page-ownership model (Component 4: Parallel Engine).
// Compile on a GPU host (Google Colab) with:
//     nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto
//
// One CUDA BLOCK owns one page. The batch is binned by page on the host (CSR offsets),
// so block p processes only page p's contiguous queries against page p's store slice.
// Different blocks touch disjoint store slots ⇒ no cross-block conflict, no store
// atomics, no delta log. Per page: single-owner (Redis). Across pages: shared-nothing
// (Dragonfly). Threads within a block stride over the page's queries.

#include "compute.hpp"
#include <cuda_runtime.h>
#include <vector>
#include <chrono>
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                                            \
    do {                                                                           \
        cudaError_t _err = (call);                                                 \
        if (_err != cudaSuccess) {                                                 \
            std::fprintf(stderr, "CUDA error %s at %s:%d -> %s\n",                 \
                         #call, __FILE__, __LINE__, cudaGetErrorString(_err));     \
            std::abort();                                                          \
        }                                                                          \
    } while (0)

// One block per page. blockIdx.x == page id; the block's threads cooperatively process
// that page's queries [offsets[page], offsets[page+1]). Writes to a slot within a page
// are owned by this block alone, so no atomics on the store are needed. Same-slot writes
// within the page race between threads -> last-writer-wins, matching the CPU mock's
// deterministic in-order result only when keys are unique (true for our benchmark).
__global__ void matrix_page_kernel(const DatabaseQuery* binned, const uint32_t* offsets,
                                   uint64_t* store,
                                   unsigned long long* reads,
                                   unsigned long long* writes) {
    const size_t page = blockIdx.x;
    if (page >= MATRIX_PAGE_COUNT) return;

    const uint32_t begin = offsets[page];
    const uint32_t end = offsets[page + 1];

    unsigned r = 0, w = 0;
    for (uint32_t j = begin + threadIdx.x; j < end; j += blockDim.x) {
        const DatabaseQuery q = binned[j];
        const size_t slot = q.query_id & MATRIX_STORE_MASK;
        if (q.opcode == OP_READ) {
            volatile uint64_t v = store[slot]; (void)v; // touch the store (read path)
            ++r;
        } else if (q.opcode == OP_WRITE) {
            store[slot] = q.query_id; // mock projection: value == key
            ++w;
        }
    }
    if (r) atomicAdd(reads, (unsigned long long)r);   // counters only — not on the store
    if (w) atomicAdd(writes, (unsigned long long)w);
}

// Filter-count scan: count values > threshold. Grid-stride over resident VRAM data,
// block-local reduction, one atomicAdd per block. This is the GPU's home turf —
// streaming bandwidth over data too large for CPU cache.
__global__ void matrix_scan_kernel(const uint64_t* data, size_t n,
                                   uint64_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        if (data[i] > threshold) ++local;
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// uint32 column variant — half the bytes/value. Same grid-stride filter-count.
__global__ void matrix_scan_kernel_u32(const uint32_t* data, size_t n,
                                       uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        if (data[i] > threshold) ++local;
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// Vectorized uint32 scan: each thread loads 4 values per instruction via uint4
// (LDG.128 = 16 bytes/load). This is the standard memory-bound fix across DL/DB/HPC
// kernels — more bytes in flight per thread => deeper memory-level parallelism =>
// closer to peak DRAM bandwidth than the scalar 4-byte load. Requires n % 4 == 0 and
// 16-byte-aligned data (cudaMalloc guarantees alignment; n is a power of two here).
__global__ void matrix_scan_kernel_u32x4(const uint4* data4, size_t n4,
                                         uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n4; i += stride) {
        const uint4 v = data4[i]; // one 16-byte load -> 4 comparisons
        local += (v.x > threshold) + (v.y > threshold)
               + (v.z > threshold) + (v.w > threshold);
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// Items-per-thread uint32 scan (register blocking) — the lever CUB's BlockReduce uses
// (128 threads x 4 items each). Each thread issues ITEMS independent loads into a
// register array BEFORE comparing any, so multiple loads are in flight per thread =>
// DRAM latency is hidden => bandwidth saturates. Unlike uint4 this needs no special
// type/alignment and doesn't inflate per-load register width. Strided access keeps the
// warp's ITEMS loads coalesced (lane L reads base+L, base+L+blockDim, ...).
template <int ITEMS>
__global__ void matrix_scan_kernel_ipt(const uint32_t* data, size_t n,
                                       uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t base_stride = (size_t)gridDim.x * blockDim.x;
    const size_t chunk = base_stride * ITEMS;
    // Striped layout: thread's global id is the base; item k is base + k*base_stride.
    // (base init must NOT include *ITEMS — that mixes the blocked convention with the
    // striped strides below and leaves most indices unvisited. Verified by
    // test_scan_coverage.cpp.)
    for (size_t base = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
         base < n; base += chunk) {
        uint32_t reg[ITEMS];
        #pragma unroll
        for (int k = 0; k < ITEMS; ++k) {
            const size_t idx = base + (size_t)k * base_stride;
            reg[k] = (idx < n) ? data[idx] : 0u; // load all ITEMS first (in flight)
        }
        #pragma unroll
        for (int k = 0; k < ITEMS; ++k) {
            const size_t idx = base + (size_t)k * base_stride;
            if (idx < n) local += (reg[k] > threshold); // then compare
        }
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

/**
 * @brief Real CUDA GPU engine, page-ownership model. Device-resident store persists
 * across batches (it is the database). Same ComputeInterface + correctness contract
 * as CPUMockEngine.
 */
class CUDAGPUEngine : public ComputeInterface {
public:
    explicit CUDAGPUEngine(size_t /*worker_count*/ = 0)
        : h_binned_(MATRIX_BATCH_MAX) {
        int device_count = 0;
        CUDA_CHECK(cudaGetDeviceCount(&device_count));
        if (device_count == 0) {
            std::fprintf(stderr, "CUDAGPUEngine: no CUDA device found.\n");
            std::abort();
        }
        cudaDeviceProp prop{};
        CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));

        CUDA_CHECK(cudaMalloc(&d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMemset(d_store_, 0, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMalloc(&d_binned_, MATRIX_BATCH_MAX * sizeof(DatabaseQuery)));
        CUDA_CHECK(cudaMalloc(&d_offsets_, (MATRIX_PAGE_COUNT + 1) * sizeof(uint32_t)));
        CUDA_CHECK(cudaMalloc(&d_reads_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMalloc(&d_writes_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_reads_, 0, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_writes_, 0, sizeof(unsigned long long)));

        // Resident analytical column (value[i]=i), filled once, scanned in place by
        // OP_SCAN. This is the GPU-DB's actual data — never shipped per query.
        CUDA_CHECK(cudaMalloc(&d_scan_col_, MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t)));
        {
            std::vector<uint32_t> h(MATRIX_SCAN_COLUMN_SIZE);
            for (size_t i = 0; i < MATRIX_SCAN_COLUMN_SIZE; ++i) h[i] = static_cast<uint32_t>(i);
            CUDA_CHECK(cudaMemcpy(d_scan_col_, h.data(),
                                  MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t),
                                  cudaMemcpyHostToDevice));
        }
        CUDA_CHECK(cudaMalloc(&d_scan_count_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_scan_count_, 0, sizeof(unsigned long long)));

        offsets_.resize(MATRIX_PAGE_COUNT + 1);

        // ponytail: single stream. Hyper-Q multi-stream is the throughput upgrade.
        CUDA_CHECK(cudaStreamCreate(&stream_));
        std::printf("CUDAGPUEngine initialized on '%s' (%d SMs, page-ownership, %zu pages).\n",
                    prop.name, prop.multiProcessorCount, MATRIX_PAGE_COUNT);
    }

    ~CUDAGPUEngine() override {
        cudaFree(d_store_);
        cudaFree(d_binned_);
        cudaFree(d_offsets_);
        cudaFree(d_reads_);
        cudaFree(d_writes_);
        cudaFree(d_scan_col_);
        cudaFree(d_scan_count_);
        cudaStreamDestroy(stream_);
        std::printf("CUDAGPUEngine released device resources.\n");
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX;

        // Bin by page on the host (folds into the dual-trigger batcher later).
        matrix_bin_by_page(batch_array, count, h_binned_.data(), offsets_.data());

        CUDA_CHECK(cudaMemcpyAsync(d_binned_, h_binned_.data(), count * sizeof(DatabaseQuery),
                                   cudaMemcpyHostToDevice, stream_));
        CUDA_CHECK(cudaMemcpyAsync(d_offsets_, offsets_.data(),
                                   (MATRIX_PAGE_COUNT + 1) * sizeof(uint32_t),
                                   cudaMemcpyHostToDevice, stream_));

        // One block per page; 128 threads/block stride over the page's queries.
        // Point ops (read/write) route by page ownership; OP_SCAN is handled below.
        constexpr int TPB = 128;
        matrix_page_kernel<<<MATRIX_PAGE_COUNT, TPB, 0, stream_>>>(
            d_binned_, d_offsets_, d_store_, d_reads_, d_writes_);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaStreamSynchronize(stream_));

        // OP_SCAN: each scan query runs the proven u32x4 kernel over the resident column.
        // ponytail: one kernel launch per scan query, sequential. Scans are rare relative
        // to point ops; fusing many predicates into one pass is the future optimization.
        for (size_t i = 0; i < count; ++i) {
            if (batch_array[i].opcode != OP_SCAN) continue;
            const auto st0 = std::chrono::steady_clock::now();
            const uint32_t threshold = matrix_get_scan_threshold(batch_array[i]);
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), stream_));
            constexpr int SCAN_TPB = 256;
            const int blocks = 1024;
            const uint4* col4 = reinterpret_cast<const uint4*>(d_scan_col_);
            matrix_scan_kernel_u32x4<<<blocks, SCAN_TPB, 0, stream_>>>(
                col4, MATRIX_SCAN_COLUMN_SIZE / 4, threshold, d_scan_count_);
            CUDA_CHECK(cudaGetLastError());
            unsigned long long c = 0;
            CUDA_CHECK(cudaMemcpyAsync(&c, d_scan_count_, sizeof(c),
                                       cudaMemcpyDeviceToHost, stream_));
            CUDA_CHECK(cudaStreamSynchronize(stream_));
            scan_time_s_ += std::chrono::duration<double>(
                std::chrono::steady_clock::now() - st0).count();
            batch_array[i].transaction_id = c;
            ++scans_;
            scan_result_sum_ += c;
        }
    }

    uint64_t reads() const override { return read_counter(d_reads_); }
    uint64_t writes() const override { return read_counter(d_writes_); }
    uint64_t commits() const override { return read_counter(d_writes_); } // every write commits (owned slot)
    uint64_t scans() const override { return scans_; }
    uint64_t scan_result_sum() const override { return scan_result_sum_; }
    double scan_time_s() const override { return scan_time_s_; }

    uint64_t store_checksum() const override {
        // ponytail: copy the whole store back (32KB) and sum on host. Once, off the
        // hot path — a device reduction would be premature for a correctness check.
        std::vector<uint64_t> h(MATRIX_STORE_SLOTS);
        CUDA_CHECK(cudaMemcpy(h.data(), d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t),
                              cudaMemcpyDeviceToHost));
        uint64_t sum = 0;
        for (uint64_t v : h) sum += v;
        return sum;
    }

    double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) override {
        return timed_scan<uint64_t>(n, threshold, out_count,
            [](const uint64_t* d, size_t nn, uint64_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel<<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

    double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) override {
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel_u32<<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

    double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // Same resident-column setup, but the kernel reads 4 u32 per uint4 load.
        // n is a power of two (>= 65536) so n % 4 == 0; cudaMalloc is 256-byte aligned.
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                const uint4* d4 = reinterpret_cast<const uint4*>(d);
                matrix_scan_kernel_u32x4<<<blocks, tpb, 0, s>>>(d4, nn / 4, thr, c);
            });
    }

    double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) override {
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel_ipt<8><<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

private:
    // Shared scan harness: alloc resident column, fill (untimed), time kernel via CUDA
    // events, return seconds. Templated on column type so u32/u64 share one code path.
    template <typename T, typename Launch>
    double timed_scan(size_t n, T threshold, uint64_t& out_count, Launch launch) {
        T* d_data = nullptr;
        unsigned long long* d_count = nullptr;
        CUDA_CHECK(cudaMalloc(&d_data, n * sizeof(T)));
        CUDA_CHECK(cudaMalloc(&d_count, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(unsigned long long)));
        {
            std::vector<T> h(n);
            for (size_t i = 0; i < n; ++i) h[i] = static_cast<T>(i);
            CUDA_CHECK(cudaMemcpy(d_data, h.data(), n * sizeof(T), cudaMemcpyHostToDevice));
        }

        constexpr int TPB = 256;
        const int blocks = 1024; // saturate the 40 SMs; grid-stride handles any n
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));
        CUDA_CHECK(cudaEventRecord(start, stream_));
        launch(d_data, n, threshold, d_count, blocks, TPB, stream_);
        CUDA_CHECK(cudaEventRecord(stop, stream_));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));

        unsigned long long h_count = 0;
        CUDA_CHECK(cudaMemcpy(&h_count, d_count, sizeof(h_count), cudaMemcpyDeviceToHost));
        out_count = h_count;

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
        cudaFree(d_data);
        cudaFree(d_count);
        return ms / 1e3; // seconds
    }

    static uint64_t read_counter(const unsigned long long* d_ptr) {
        unsigned long long h = 0;
        CUDA_CHECK(cudaMemcpy(&h, d_ptr, sizeof(h), cudaMemcpyDeviceToHost));
        return static_cast<uint64_t>(h);
    }

    uint64_t* d_store_ = nullptr;
    DatabaseQuery* d_binned_ = nullptr;
    uint32_t* d_offsets_ = nullptr;
    unsigned long long* d_reads_ = nullptr;
    unsigned long long* d_writes_ = nullptr;
    uint32_t* d_scan_col_ = nullptr;            // resident analytical column
    unsigned long long* d_scan_count_ = nullptr; // scratch for one scan's count
    uint64_t scans_ = 0;                          // host-side scan accounting
    uint64_t scan_result_sum_ = 0;
    double scan_time_s_ = 0.0;
    std::vector<DatabaseQuery> h_binned_;
    std::vector<uint32_t> offsets_;
    cudaStream_t stream_{};
};


In [ ]:
%%writefile main.cpp
#include "types.hpp"
#include "ring_buffer.hpp"
#if defined(MATRIX_USE_CUDA)
    #include "compute_cuda.cuh"   // real GPU engine (build with nvcc -DMATRIX_USE_CUDA)
    using EngineType = CUDAGPUEngine;
#else
    #include "compute_mock.cpp"   // CPU fallback (default; runs anywhere)
    using EngineType = CPUMockEngine;
#endif
#include <iostream>
#include <chrono>
#include <thread>
#include <memory>
#include <array>
#include <atomic>
#include <cassert>
#include <vector>
#include <algorithm>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    #include <pthread.h>
    #include <pthread/qos.h>        // ponytail: spec omitted these 3; needed to compile the
    #include <mach/mach.h>          // QoS + Mach affinity calls below on macOS
    #include <mach/thread_policy.h>
#elif defined(__linux__)
    #include <sched.h>
#endif

/**
 * @brief Platform-agnostic pipeline stalling barrier for low-overhead busy spins.
 */
inline void spin_stall() noexcept {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Instruction Synchronization Barrier (isb) forces instruction execution pipeline flush on ARM64
    asm volatile("isb" ::: "memory");
#elif defined(__x86_64__)
    // Standard pause execution hint
    asm volatile("pause" ::: "memory");
#else
    asm volatile("" ::: "memory");
#endif
}

/**
 * @brief Configures OS-specific thread-pinning and priority policies to ensure execution on performance cores.
 */
inline void pin_to_performance_core() {
#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Promotes the calling thread to the interactive QoS class to guarantee performance core placement on macOS
    pthread_set_qos_class_self_np(QOS_CLASS_USER_INTERACTIVE, 0);

    // Advisory affinity grouping configuration on Apple Silicon Mach kernels
    thread_port_t mach_thread = pthread_mach_thread_np(pthread_self());
    thread_affinity_policy_data_t policy = { 1 };
    thread_policy_set(mach_thread, THREAD_AFFINITY_POLICY, (thread_policy_t)&policy, THREAD_AFFINITY_POLICY_COUNT);
#elif defined(__linux__)
    cpu_set_t cpuset;
    CPU_ZERO(&cpuset);
    CPU_SET(0, &cpuset); // Pin strictly to logical Core 0
    pthread_setaffinity_np(pthread_self(), sizeof(cpu_set_t), &cpuset);
#endif
}

/**
 * @brief Raw SPSC handoff latency via ping-pong: the producer pushes exactly one
 * item, waits until the consumer has popped it, then pushes the next. This isolates
 * the pure enqueue->dequeue cost — the spec's "sub-microsecond" claim — with zero
 * queue backlog, unlike the pipeline's queue-residency number which is dominated by
 * burst backlog. Backend-independent: measures only the ring buffer.
 */
void measure_handoff_latency() {
    constexpr size_t WARMUP = 1000;   // discard cold-cache / thread-migration samples
    constexpr size_t SAMPLES = 100000;
    auto ring = std::make_unique<SPSCRingBuffer<DatabaseQuery, 4096>>();
    std::atomic<bool> running{true};
    std::vector<uint64_t> samples;
    samples.reserve(SAMPLES);

    // Consumer pops and timestamps; it is the sole writer of `samples` (joined before read).
    std::thread consumer([&]() {
        pin_to_performance_core();
        DatabaseQuery v;
        while (running.load(std::memory_order_relaxed)) {
            if (ring->try_pop(v)) {
                const uint64_t now_ns = static_cast<uint64_t>(
                    std::chrono::steady_clock::now().time_since_epoch().count());
                if (v.query_id >= WARMUP) {
                    samples.push_back(now_ns - v.timestamp_us);
                }
            }
        }
    });

    for (size_t i = 0; i < WARMUP + SAMPLES; ++i) {
        DatabaseQuery q{};
        q.query_id = i;
        q.timestamp_us = static_cast<uint64_t>(
            std::chrono::steady_clock::now().time_since_epoch().count());
        while (!ring->try_emplace(q)) spin_stall();
        while (!ring->empty()) spin_stall(); // wait for consumer to take it (ping-pong)
    }
    running.store(false);
    consumer.join();

    std::sort(samples.begin(), samples.end());
    const auto pct = [&](double p) { return samples[static_cast<size_t>(p * (samples.size() - 1))]; };
    std::cout << "Raw SPSC handoff (ns)    "
              << "p50=" << pct(0.50) << "  p99=" << pct(0.99)
              << "  p99.9=" << pct(0.999) << "  max=" << samples.back() << std::endl;
}

/**
 * @brief Throughput vs. batch size sweep — the key question for GPU viability.
 * Calls execute_batch() directly (no ring) so it measures pure engine cost per query
 * across batch sizes. Reveals fixed per-batch overhead and where (if ever) larger
 * batches amortize it. One run gives the whole curve — precious when GPU runs are remote.
 */
void sweep_batch_sizes(ComputeInterface& engine) {
    constexpr size_t QUERIES_PER_POINT = 400000; // enough work to dwarf timer noise
    const size_t sizes[] = {64, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536};

    std::cout << "--- Batch-size sweep (execute_batch only, " << QUERIES_PER_POINT
              << " queries/point) ---" << std::endl;

    std::vector<DatabaseQuery> batch(MATRIX_BATCH_MAX);
    for (size_t i = 0; i < MATRIX_BATCH_MAX; ++i) {
        batch[i] = DatabaseQuery{};
        batch[i].query_id = i;
        batch[i].opcode = (i % 2 == 0) ? OP_READ : OP_WRITE;
    }

    // Warm up the engine (first GPU call pays one-time CUDA init) so it doesn't
    // pollute the smallest batch point.
    engine.execute_batch(batch.data(), 256);

    for (size_t bs : sizes) {
        const size_t iters = QUERIES_PER_POINT / bs;
        const auto t0 = std::chrono::steady_clock::now();
        for (size_t it = 0; it < iters; ++it) {
            engine.execute_batch(batch.data(), bs);
        }
        const auto t1 = std::chrono::steady_clock::now();
        const double secs = std::chrono::duration<double>(t1 - t0).count();
        const double ops = static_cast<double>(iters * bs) / secs;
        const double us_per_batch = (secs / iters) * 1e6;
        std::cout << "  batch=" << bs
                  << "\t" << static_cast<uint64_t>(ops) << " ops/sec"
                  << "\t" << us_per_batch << " us/batch" << std::endl;
    }
}

/**
 * @brief Scan benchmark — the GPU's home turf. Filter-count over RESIDENT data of
 * growing size: small (fits CPU cache) to large (far exceeds it). The crossover where
 * GPU overtakes CPU is the whole point — exploiting bandwidth over big data, not
 * shipping tiny point-ops over PCIe. Verifies the count against a known oracle.
 */
void scan_benchmark(ComputeInterface& engine) {
    const size_t sizes[] = {1u<<16, 1u<<18, 1u<<20, 1u<<22, 1u<<24, 1u<<26}; // 64K .. 64M values
    std::cout << "--- Scan benchmark (filter-count over resident data) ---" << std::endl;
    for (size_t n : sizes) {
        const uint64_t threshold = n / 2;
        uint64_t c64 = 0, c32 = 0, c32x4 = 0, cipt = 0;
        const double s64 = engine.benchmark_scan(n, threshold, c64);
        const double s32 = engine.benchmark_scan_u32(n, static_cast<uint32_t>(threshold), c32);
        const double s32x4 = engine.benchmark_scan_u32x4(n, static_cast<uint32_t>(threshold), c32x4);
        const double sipt = engine.benchmark_scan_ipt(n, static_cast<uint32_t>(threshold), cipt);
        // value[i]=i, so count of i>threshold == n-1-threshold (oracle).
        assert(c64 == n - 1 - threshold && "u64 scan produced wrong count");
        assert(c32 == n - 1 - threshold && "u32 scan produced wrong count");
        assert(c32x4 == n - 1 - threshold && "u32x4 vectorized scan produced wrong count");
        assert(cipt == n - 1 - threshold && "items-per-thread scan produced wrong count");
        std::cout << "  n=" << n
                  << "\tu64 " << (n * sizeof(uint64_t)) / 1e9 / s64 << " GB/s"
                  << "\tu32 " << (n * sizeof(uint32_t)) / 1e9 / s32 << " GB/s"
                  << "\tu32x4 " << (n * sizeof(uint32_t)) / 1e9 / s32x4 << " GB/s"
                  << "\tipt8 " << (n * sizeof(uint32_t)) / 1e9 / sipt << " GB/s"
                  << " (" << static_cast<uint64_t>(n / sipt / 1e6) << "M vals/s)"
                  << std::endl;
    }
}

int main() {
    std::cout << "MatrixDB Bare-Metal Engine Booting..." << std::endl;

    // Microbenchmark first: pure ring handoff, before the pipeline saturates anything.
    measure_handoff_latency();

    constexpr size_t BATCH_SIZE_LIMIT = 512;
    constexpr uint64_t TEMPORAL_LIMIT_US = 50;
    constexpr size_t TOTAL_QUERIES = 10000;

    auto ring_buffer = std::make_unique<SPSCRingBuffer<DatabaseQuery, 4096>>();
    auto mock_engine = std::make_unique<EngineType>(4);

    // Sweep batch sizes before the pipeline run so one execution yields the whole
    // throughput-vs-batch curve (decisive for GPU viability; remote runs are costly).
    sweep_batch_sizes(*mock_engine);
    scan_benchmark(*mock_engine);

    std::atomic<bool> run_state{true};
    std::atomic<size_t> total_processed{0}; // ponytail: end-to-end check that every query flows through

    // Pre-reserved so the hot path never allocates. Filled by the single consumer only.
    std::vector<uint64_t> latencies_ns;
    latencies_ns.reserve(TOTAL_QUERIES);

    // Spin up the consumer thread to execute the ingestion busy-spin control loop
    std::thread consumer([&]() {
        pin_to_performance_core();

        std::array<DatabaseQuery, BATCH_SIZE_LIMIT> batch;
        size_t accumulated_queries = 0;
        auto batch_start_time = std::chrono::high_resolution_clock::now();

        while (run_state.load(std::memory_order_relaxed)) {
            DatabaseQuery incoming_query;
            const bool success = ring_buffer->try_pop(incoming_query);

            if (success) {
                // Queue residency = time from producer's enqueue stamp to this pop.
                const uint64_t now_ns = static_cast<uint64_t>(
                    std::chrono::steady_clock::now().time_since_epoch().count());
                latencies_ns.push_back(now_ns - incoming_query.timestamp_us);

                if (accumulated_queries == 0) {
                    batch_start_time = std::chrono::high_resolution_clock::now();
                }
                batch[accumulated_queries++] = incoming_query;
            }

            // Continuous evaluation of the Dual-Trigger Condition
            if (accumulated_queries > 0) {
                const auto current_time = std::chrono::high_resolution_clock::now();
                const auto duration_us = std::chrono::duration_cast<std::chrono::microseconds>(
                    current_time - batch_start_time
                ).count();

                // Trigger flush if batch is full OR time-based window has expired
                if (accumulated_queries >= BATCH_SIZE_LIMIT || duration_us >= static_cast<long long>(TEMPORAL_LIMIT_US)) {
                    mock_engine->execute_batch(batch.data(), accumulated_queries);
                    total_processed.fetch_add(accumulated_queries, std::memory_order_relaxed);
                    accumulated_queries = 0;
                }
            }

            if (!success) {
                spin_stall();
            }
        }
    });

    // Snapshot engine counters after the sweep so the pipeline asserts below measure
    // only the pipeline's 10k queries, not the sweep's traffic.
    const uint64_t reads_base = mock_engine->reads();
    const uint64_t writes_base = mock_engine->writes();
    const uint64_t applied_base = mock_engine->commits();
    const uint64_t scans_base = mock_engine->scans();
    const uint64_t scan_sum_base = mock_engine->scan_result_sum();

    // Every Nth query is an analytical OP_SCAN with a fixed threshold; the rest are
    // point ops. Lets us prove scans flow through the same ring + batcher to the engine.
    constexpr size_t SCAN_EVERY = 1000;
    constexpr uint32_t SCAN_THRESHOLD = MATRIX_SCAN_COLUMN_SIZE / 2;
    size_t expected_scans = 0;
    for (size_t i = 0; i < TOTAL_QUERIES; ++i) if (i % SCAN_EVERY == 0) ++expected_scans;
    // Column is value[j]=j, so each scan counts (SIZE-1-threshold); sum = scans * that.
    const uint64_t expected_scan_sum =
        static_cast<uint64_t>(expected_scans) * (MATRIX_SCAN_COLUMN_SIZE - 1 - SCAN_THRESHOLD);

    // Simulate query production on a separate thread
    auto pipeline_start = std::chrono::steady_clock::now();
    std::thread producer([&]() {
        for (size_t i = 0; i < TOTAL_QUERIES; ++i) {
            // Stamp ingestion time (steady_clock ns) so the consumer can measure queue residency.
            const uint64_t stamp_ns = static_cast<uint64_t>(
                std::chrono::steady_clock::now().time_since_epoch().count());
            DatabaseQuery q{i, stamp_ns, 0, OP_READ, 8, {0}, {0}};
            if (i % SCAN_EVERY == 0) {
                matrix_set_scan_threshold(q, SCAN_THRESHOLD); // analytical scan query
            } else {
                q.opcode = (i % 2 == 0) ? OP_READ : OP_WRITE; // point ops otherwise
            }
            while (!ring_buffer->try_emplace(q)) {
                spin_stall();
            }
        }
    });

    producer.join();

    // Wait until every produced query has been processed, then stamp the drain time.
    // Replaces an arbitrary fixed sleep so throughput reflects real drain, not slack.
    while (total_processed.load(std::memory_order_relaxed) < TOTAL_QUERIES) {
        spin_stall();
    }
    auto pipeline_end = std::chrono::steady_clock::now();
    run_state.store(false);

    if (consumer.joinable()) {
        consumer.join();
    }

    const double elapsed_s =
        std::chrono::duration<double>(pipeline_end - pipeline_start).count();

    const size_t processed = total_processed.load();
    std::cout << "Processed " << processed << " / " << TOTAL_QUERIES << " queries." << std::endl;
    assert(processed == TOTAL_QUERIES && "Dual-trigger pipeline dropped queries");

    // Verify the engine actually dispatched on opcode and committed every mutation.
    // Deltas since the pre-pipeline snapshot isolate the pipeline from the sweep.
    const uint64_t reads = mock_engine->reads() - reads_base;
    const uint64_t writes = mock_engine->writes() - writes_base;
    const uint64_t applied = mock_engine->commits() - applied_base;
    const uint64_t scans = mock_engine->scans() - scans_base;
    const uint64_t scan_sum = mock_engine->scan_result_sum() - scan_sum_base;
    std::cout << "Engine: reads=" << reads << " writes=" << writes
              << " commits=" << applied << " scans=" << scans << std::endl;
    std::cout << "Store checksum: " << mock_engine->store_checksum()
              << " (must match across CPU and GPU backends)" << std::endl;
    std::cout << "Scan result sum: " << scan_sum
              << " (oracle " << expected_scan_sum << ")" << std::endl;
    assert(reads + writes + scans == processed && "opcode dispatch missed queries");
    assert(applied == writes && "page-ownership commit dropped mutations");
    assert(scans == expected_scans && "scan queries dropped in pipeline");
    assert(scan_sum == expected_scan_sum && "GPU scan result mismatch vs oracle");

    // Report throughput split by workload. A single 64MB scan costs ~1000x a point op,
    // so a combined ops/sec is meaningless once scans are mixed in (HTAP: OLTP point ops
    // vs OLAP scans on one path). Separate them: point-op rate excludes scan time.
    const double scan_s = mock_engine->scan_time_s();
    const double pointop_s = elapsed_s - scan_s;
    const uint64_t point_ops = reads + writes;
    std::cout << "Throughput: point-ops " << static_cast<uint64_t>(point_ops / pointop_s)
              << " ops/sec (" << point_ops << " in " << pointop_s * 1e3 << " ms)";
    if (scans > 0) {
        std::cout << " | scans " << static_cast<uint64_t>(scans / scan_s)
                  << " scans/sec (" << scans << " in " << scan_s * 1e3 << " ms, "
                  << scan_s / scans * 1e3 << " ms/scan)";
    }
    std::cout << std::endl;

    // Queue residency under burst: time each query waits in the ring before it is
    // batched. Dominated by backlog (producer outruns the single consumer), so this
    // is NOT the raw handoff cost reported above — it is the saturated-pipeline view.
    if (!latencies_ns.empty()) {
        std::sort(latencies_ns.begin(), latencies_ns.end());
        const auto pct = [&](double p) {
            const size_t idx = static_cast<size_t>(p * (latencies_ns.size() - 1));
            return latencies_ns[idx];
        };
        std::cout << "Queue residency (ns)     "
                  << "p50=" << pct(0.50) << "  "
                  << "p99=" << pct(0.99) << "  "
                  << "p99.9=" << pct(0.999) << "  "
                  << "max=" << latencies_ns.back() << std::endl;
    }

    std::cout << "Engine execution loop completed successfully." << std::endl;
    return 0;
}


In [ ]:
%%writefile test_scan_coverage.cpp
// Coverage test for the items-per-thread scan kernel's index arithmetic.
// The kernel's index math is pure integer logic — hardware-independent — so we can
// simulate it on the CPU and verify every index in [0,n) is visited exactly once.
// This is the test that WOULD have caught the GPU-only undercount bug.
//
// Build: clang++ -std=c++17 -O2 test_scan_coverage.cpp -o /tmp/tcov && /tmp/tcov
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <vector>

// Simulate the kernel's visited-index set for a given base-init convention.
// blocked_init=true reproduces the BUG (base = blockIdx*blockDim*ITEMS + tid);
// blocked_init=false is the FIX (base = blockIdx*blockDim + tid).
static bool coverage_ok(size_t n, int GRID, int TPB, int ITEMS, bool blocked_init) {
    std::vector<int> visit(n, 0);
    const size_t base_stride = (size_t)GRID * TPB;
    const size_t chunk = base_stride * ITEMS;
    for (int b = 0; b < GRID; ++b) {
        for (int x = 0; x < TPB; ++x) {
            size_t base = blocked_init
                ? (size_t)b * TPB * ITEMS + x
                : (size_t)b * TPB + x;
            for (; base < n; base += chunk) {
                for (int k = 0; k < ITEMS; ++k) {
                    const size_t idx = base + (size_t)k * base_stride;
                    if (idx < n) visit[idx]++;
                }
            }
        }
    }
    size_t missed = 0, dup = 0;
    for (size_t i = 0; i < n; ++i) {
        if (visit[i] == 0) ++missed;
        else if (visit[i] > 1) ++dup;
    }
    printf("  n=%zu init=%s  missed=%zu dup=%zu  -> %s\n",
           n, blocked_init ? "blocked(BUG)" : "striped(FIX)", missed, dup,
           (missed == 0 && dup == 0) ? "OK" : "BROKEN");
    return missed == 0 && dup == 0;
}

int main() {
    const int GRID = 1024, TPB = 256, ITEMS = 8; // exactly what the engine launches
    const size_t sizes[] = {65536, 262144, 1048576, 4194304, 67108864};

    printf("Buggy (blocked init) — expect BROKEN:\n");
    bool any_bug_broken = false;
    for (size_t n : sizes) any_bug_broken |= !coverage_ok(n, GRID, TPB, ITEMS, true);

    printf("Fixed (striped init) — expect OK:\n");
    bool all_fix_ok = true;
    for (size_t n : sizes) all_fix_ok &= coverage_ok(n, GRID, TPB, ITEMS, false);

    if (!any_bug_broken) { printf("FAIL: bug did not reproduce\n"); return 1; }
    if (!all_fix_ok)     { printf("FAIL: fix does not give exact coverage\n"); return 1; }
    printf("PASS: bug reproduced, fix gives exact coverage\n");
    return 0;
}


## 3. Kernel index-coverage test (CPU, no GPU needed)

Simulates the items-per-thread scan kernel's index arithmetic and asserts every index is visited exactly once. This is pure integer logic, so it runs on the CPU and catches GPU-only coverage bugs before spending a GPU build.

In [ ]:
!g++ -std=c++17 -O2 test_scan_coverage.cpp -o /tmp/tcov && /tmp/tcov

## 4. Build & run on the GPU

`-x cu` compiles `main.cpp` as CUDA so the `.cuh` kernels link in. `-D_GNU_SOURCE` exposes Linux thread-affinity APIs; `-Xcompiler -pthread` links std::thread.

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto

In [ ]:
!./matrixdb_proto

## 5. CPU fallback (run this if no GPU runtime is selected)

Same code, same asserts, no CUDA — proves the logic without a GPU.

In [ ]:
!g++ -std=c++17 -O3 -D_GNU_SOURCE -pthread main.cpp -o matrixdb_cpu && ./matrixdb_cpu